# [Re] Buda et al. 2019 — TCGA-LGG (single-notebook **full** replication)

Minimum-compute path to a *solid* ReScience replication. The paper's headline claim is
**not** the segmentation Dice — it is a **radiogenomics** result: shape features of the
automatically segmented tumor are associated with genomic subtype, and the deep-learning
masks preserve those associations about as well as the manual masks. Reproducing that is
CPU-only statistics on masks we already predict, so a full replication costs almost the
same GPU as segmentation alone.

**Core pipeline (this notebook, ~3–5 GPU-hours):**
1. leakage test (must pass first, R1)  2. build manifest  3. patient-level 5-fold splits
4. train the plain 4-level U-Net from scratch (R4), resumable  5. **per-patient** Dice vs
paper 0.82/0.85 (R2)  6. **shape features + radiogenomics** on predicted *and* ground-truth
masks (Section 7 — the load-bearing full-replication piece, ~0 GPU)  7. figures F1–F9
(F1–F7 required, incl. the automatic-vs-manual discrimination ROC) and
`reports/comparison.md` + `metrics.json`.

The GPU-heavy extras (robustness / quantization / distillation / cross-institution / nnU-Net)
are **not required for acceptance** and live in the optional appendix at the bottom, default
**off**.

**Setup on Kaggle:** Add Input → `mateuszbuda/lgg-mri-segmentation`; enable GPU (P100 or T4).
Everything writes to `/kaggle/working`, survives *Save & Run All*, and resumes across the
12-hour cap — just run again to continue training. Set `SMOKE=True` first to verify wiring
in ~5 min.

**Provenance:** this is a US NCI / TCIA collection from five US institutions. It is **not** an
African dataset. The `race`/`ethnicity` columns are US-population demographics used only for an
honest per-subgroup fairness figure (F9).

In [ ]:
# ============================ KNOBS ============================
SMOKE          = False      # True = 2-patient wiring check (~5 min), ignores everything below
PRESET         = 'kfold5'   # 'kfold5' (default, ~3-5 GPU-h) or 'buda22' (paper-exact 22x5, ~3x compute)
EPOCHS         = 60         # per fold; lower to fit a single session, --resume continues
IN_CHANNELS    = 3          # 3 = pre/FLAIR/post (headline, R9); 1 = FLAIR-only
BATCHNORM      = True       # True = public-repo variant; False = strictly-paper net
FOLDS_TO_TRAIN = None       # None = all folds; or e.g. [0,1] to split across sessions

# Section-12 GPU extras — NOT required for ReScience acceptance. Leave off for the core run.
RUN_OPTIONAL_EXTRAS = False

# GitHub repo with the src/lgg package (used only if the code isn't already alongside the
# notebook or added as a Kaggle dataset). REPO_BRANCH must be a branch that has been PUSHED
# to this remote and contains src/lgg + tests/. The full-replication pipeline lives on the
# feature branch of this fork (the upstream repo's main does NOT have it).
REPO_URL    = 'https://github.com/spraldev/mlrc-tracks-TCGA-LGG-comm.git'
REPO_BRANCH = 'faithful-unet-reproduction'
# Requires Internet = On in the notebook settings. Alternatively add the repo as a Kaggle
# dataset input (with src/lgg inside) and it will be auto-detected without any clone.
# ===============================================================

In [ ]:
import os, sys, glob, subprocess, shutil, json

# --- locate the src/lgg package: prefer local / a Kaggle dataset, else clone ----
def find_repo():
    cands = ['.', '/kaggle/working/repo', os.path.dirname(os.getcwd())]
    # also scan /kaggle/input in case the repo was added as a dataset (offline path)
    if os.path.isdir('/kaggle/input'):
        for dirpath, _dirnames, _files in os.walk('/kaggle/input'):
            if os.path.isdir(os.path.join(dirpath, 'src', 'lgg')):
                cands.append(dirpath)
    for c in cands:
        if os.path.isdir(os.path.join(c, 'src', 'lgg')):
            return os.path.abspath(c)
    return None

REPO = find_repo()
if REPO is None:
    dst = '/kaggle/working/repo'
    # re-clone if a previous run left a stale/incomplete checkout (no src/lgg)
    if os.path.isdir(dst) and not os.path.isdir(os.path.join(dst, 'src', 'lgg')):
        shutil.rmtree(dst)
    if not os.path.isdir(dst):
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, dst],
                       check=True)
    REPO = dst
# hard fail early with a clear message if the code still isn't here
if not os.path.isdir(os.path.join(REPO, 'src', 'lgg')):
    raise FileNotFoundError(
        f"src/lgg not found under {REPO}. The clone of branch '{REPO_BRANCH}' did not contain "
        "the pipeline. Push that branch to the remote (with src/lgg + tests/), or add the repo "
        "as a Kaggle dataset input.")
print('repo:', REPO, '| branch target:', REPO_BRANCH)

# --- deps: Kaggle ships torch/pandas/scipy/sklearn; add medpy for HD95 ---------
try:
    import medpy  # noqa
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'medpy'], check=False)

SRC = os.path.join(REPO, 'src')
sys.path.insert(0, SRC)

def run(*cli_args):
    """Invoke the lgg CLI with the working config, streaming output live."""
    env = dict(os.environ, PYTHONPATH=SRC)
    cmd = [sys.executable, '-m', 'lgg.cli', '--config', CONFIG] + list(cli_args)
    print('>>>', ' '.join(cli_args))
    p = subprocess.run(cmd, env=env, cwd=WORK)
    if p.returncode != 0:
        raise RuntimeError(f'command failed: {cli_args}')

In [ ]:
# --- working dir + auto-detected dataset path + generated config.yaml ----------
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(REPO, '_run')
os.makedirs(WORK, exist_ok=True)

def detect_datapath():
    # Find the folder that directly contains TCGA_* patient subfolders.
    roots = ['/kaggle/input', os.path.join(REPO, 'data')]
    for root in roots:
        for dirpath, dirnames, _ in os.walk(root):
            if any(d.startswith('TCGA_') and os.path.isdir(os.path.join(dirpath, d)) for d in dirnames):
                return dirpath
    raise FileNotFoundError('Could not find TCGA_* folders. Add the lgg-mri-segmentation dataset.')

DATAPATH = detect_datapath()
print('datapath:', DATAPATH)
# data.csv (genomic + demographic labels) ships with the dataset; radiogenomics auto-discovers it.
print('data.csv found:', bool(glob.glob(os.path.join(os.path.dirname(DATAPATH), '**', 'data.csv'), recursive=True))
      or os.path.exists(os.path.join(DATAPATH, 'data.csv')))

CONFIG = os.path.join(WORK, 'config.yaml')
cfg = f'''seed: 42
paths:
  datapath: {DATAPATH}
  manifest: {WORK}/reports/manifest.csv
  splits: {WORK}/splits/splits.json
  ckpt_dir: {WORK}/checkpoints
  reports: {WORK}/reports
  figures: {WORK}/figures
data:
  in_channels: {IN_CHANNELS}
splits:
  preset: {PRESET}
  seed: 42
  stratify_by_site: false
train:
  epochs: {EPOCHS}
  batch_size: 16
  lr: 0.001
  channels: reference
  batchnorm: {str(BATCHNORM).lower()}
  augment: true
  amp: true
  num_workers: 2
  foreground_bias: 0.0
  dice_weight: 1.0
  bce_weight: 1.0
  grad_clip: 0.0
  pretrained_encoder: false
distill:
  student_channels: micro
  epochs: 30
  temperature: 2.0
  alpha: 0.5
'''
with open(CONFIG, 'w') as fh:
    fh.write(cfg)
print(cfg)

## R1 — leakage test first
Patient-level splits: no patient in two folds; val sets pairwise disjoint and covering all 110.
This must pass before any training runs.

In [ ]:
env = dict(os.environ, PYTHONPATH=SRC)
subprocess.run([sys.executable, '-m', 'pytest', os.path.join(REPO, 'tests', 'test_no_leakage.py'), '-q'],
               env=env, cwd=REPO, check=True)

In [ ]:
# --- SMOKE path: 2-patient end-to-end wiring check, then stop -----------------
if SMOKE:
    run('prepare-data')
    run('smoke')
    raise SystemExit('SMOKE OK — set SMOKE=False for the full run.')

## Steps 1–3 — manifest, patient-level splits, train from scratch
Re-running the training cell after a session dies continues each fold from its last epoch
checkpoint (`--resume`); finished folds return immediately.

In [ ]:
run('prepare-data')
run('make-splits')

In [ ]:
from lgg.data.splits import load_splits
folds = load_splits(f'{WORK}/splits/splits.json')['folds']
targets = FOLDS_TO_TRAIN if FOLDS_TO_TRAIN is not None else [f['fold'] for f in folds]
for fi in targets:
    run('train', '--fold', str(fi), '--resume')

## Step 4 — HEADLINE per-patient Dice/IoU/HD95 vs paper 0.82/0.85 (R2, R3)

In [ ]:
run('evaluate')

## Step 5 — Radiogenomics (Section 7, the full-replication piece, ~0 GPU)
`shape-features` reconstructs each patient's 3D mask volume (predicted **and** ground truth),
extracts BEVR / angular-std / margin-fluctuation, and joins the genomic + demographic labels
from `data.csv`. `radiogenomics` runs Bonferroni-corrected Fisher tests and the cluster-vs-rest
discrimination AUC for both mask sources — reproducing the paper's actual headline claim.

In [ ]:
run('shape-features')
run('radiogenomics')

## Step 6 — figures F1–F9 + reports (metrics.json + comparison.md)
F1–F7 are required (incl. F7, the predicted-vs-manual discrimination ROC — the paper's
headline validation); F8 (shape-feature agreement) and F9 (fairness) are optional but free.

In [ ]:
run('figures')
run('report')

In [ ]:
# --- show the headline + radiogenomics comparison and every figure inline -----
from IPython.display import Markdown, display
import matplotlib.pyplot as plt, matplotlib.image as mpimg
display(Markdown(open(f'{WORK}/reports/comparison.md').read()))
for fp in sorted(glob.glob(f'{WORK}/figures/*.png')):
    plt.figure(figsize=(9, 5)); plt.imshow(mpimg.imread(fp)); plt.axis('off')
    plt.title(os.path.basename(fp)); plt.show()

In [ ]:
# --- bundle everything a co-author needs to write the paper ------------------
bundle = f'{WORK}/paper_artifacts'
os.makedirs(bundle, exist_ok=True)
for sub in ['reports', 'figures', 'splits']:
    if os.path.isdir(f'{WORK}/{sub}'):
        shutil.copytree(f'{WORK}/{sub}', f'{bundle}/{sub}', dirs_exist_ok=True)
shutil.make_archive(f'{WORK}/paper_artifacts', 'zip', bundle)
print('Wrote', f'{WORK}/paper_artifacts.zip')
for f in sorted(glob.glob(f'{bundle}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(' ', os.path.relpath(f, bundle))

---
## Optional appendix — Section 12 GPU extras (NOT required for ReScience acceptance)
These add GPU hours and do not affect acceptance (ReScience does not judge novelty). They reuse
the same leakage-safe splits. Run only with spare quota by setting `RUN_OPTIONAL_EXTRAS = True`
in the KNOBS cell. `report` is re-run afterwards so the extras fold into `metrics.json`.

In [ ]:
if RUN_OPTIONAL_EXTRAS:
    run('run-robustness')     # inference-only Dice degradation under noise/downsample/bias/dropout
    run('quantize')           # FP16 + INT8 PTQ: size/latency/Dice delta, GPU + CPU
    run('distill')            # reduced-channel student from teacher outputs
    run('cross-institution')  # leave-one-institution-out generalization (retrains per site)
    run('report')             # re-assemble metrics.json + comparison.md with the extras
    from IPython.display import Markdown, display
    display(Markdown(open(f'{WORK}/reports/comparison.md').read()))
else:
    print('Optional extras skipped (RUN_OPTIONAL_EXTRAS=False). Core full replication is complete.')